<a href="https://colab.research.google.com/github/KarimBekhtiGalvao/Diffusion-Based-Generation/blob/main/Diffusion_Based_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Report - Diffusion Based Text Generation
##**Subject chosen:**
Most modern text generation systems are based on autoregressive Transformer models, which generate text token by token in a left-to-right fashion. While that turned out to be highly effective, this paradigm is not the only possible approach to sequence generation. In other domains, such as image, speech, and audio generation, diffusion models have shown remarkable performance by generating data through an iterative denoising process rather than autoregressive decoding.

Diffusion models generate samples by progressively refining a noisy signal into a final clean output. Instead of predicting the next token, they learn to reverse a noise corruption process, iteratively improving the quality of the generated sample. This non-autoregressive generation process offers potential advantages, such as parallel generation.

Despite their success in other modalities, diffusion-based approaches have so far shown limited effectiveness in text generation, mainly due to the discrete nature of text and the difficulty of defining suitable diffusion processes over token sequences. Nevertheless, recent work suggests that diffusion models may be potentially viable.

##**Project Objective**
The goal of this project is to design, implement, and evaluate diffusion-based models for text generation, and to compare their behavior and performance with standard autoregressive Transformer models. The focus of the project is not on achieving state-of-the-art performance, but on understanding the strengths and limitations of diffusion models for text, analyzing their training dynamics and generation behavior, and comparing them fairly to autoregressive baselines at a small scale.

##**Proposed Project Plan**

Literature Review: Conduct a focused literature review on diffusion models for sequence and text generation. In particular, try to understand the general principles of diffusion and denoising-based generative models, have insights into existing approaches to diffusion over discrete sequences or text representations, and figure out the challenges and limitations of diffusion-based text generation.

Data and Representation: Select a suitable text dataset for small-scale experimentation. Possible choices include a subset of Wikipedia or book corpora. You may operate at the character, byte, or token level. Dataset size should be chosen to keep training computationally manageable.

Baseline Model (Autoregressive): Implement a standard autoregressive text generation baseline. Train and evaluate the model using standard text generation metrics. Tune hyperparameters using validation data.

Diffusion-Based Model: Implement one or more diffusion-based models for text generation. Define an appropriate forward noise process and a reverse denoising model. Clearly explain architectural and modeling choices. Ensure that training and inference are computationally feasible.

Evaluation and Analysis: Compare diffusion-based models and autoregressive models using quantitative metrics (e.g., perplexity, negative log-likelihood, or other suitable measures). Do a qualitative analysis of the generated text, training stability, generation speed, and other relevant aspects you might find relevant. Optionally, you may explore scaling behavior, such as how performance changes with model size or dataset size for diffusion-based versus autoregressive models.



# Litterature Review

## Introduction

As seen during the COMP6861 Concordia course, generating text using Machine Learning is a complex problem, with the modeling of high-dimensional probability distributions over sequences of tokens. Approaches seen in class rely on autoregressive language models that generate text by predicting the next token using previously generated tokens. While these models have achieved strong performances such as that of Claude or ChatGPT 5, they suffer from problems such as **exposure bias**, **limited controllability**, and **difficulties modeling long-range dependencies**.

More recently, diffusion-based generative models have emerged as a promising alternative paradigm for generative modeling [Deep Unsupervised Learning using Nonequilibrium Thermodynamics, Sohl-Dickstein et al]. Used initially to generate images, diffusion models learn to make sentences by gradually transforming noise into structured data through a sequence of denoising steps. The process typically works in two stages: a forward process, which uses training data and noises it, and a denoising process, where a neural network learns to remove noise and reconstruct the initial data distribution [Diffusion models in text generation: a survey].

Diffusion models have been successful in image generation so recent research has begun adapting them to discrete data, including natural language. Applying diffusion models to text introduces issues because of the discrete nature of tokens/text and the difficulty of defining continuous noise processes over discrete representations. To solve these problems, several approaches have been proposed, including continuous embeddings [A CHEAPER AND BETTER DIFFUSION LANGUAGE MODEL WITH SOFT-MASKED NOISE], discrete diffusion processes, and hybrid autoregressive–diffusion architectures [AR-DIFFUSION: Auto-Regressive Diffusion Model for Text Generation].

This line of research shows that diffusion-based generation may offer advantages such as diversity, better coherence, and more flexible generation compared to traditional models. Let's dive by using the survey [Diffusion models in text generation: a survey] which provides a timelime of recent influential models : [Diffusion-NAT], [AR-DIFFUSION], [DiffuSum], [Masked-Diffuse LM], [DiNoiSer] and [RDMs].

# Domain Inception

In [Deep Unsupervised Learning using Nonequilibrium Thermodynamics], Sohl-Dickstein et al launched the domain of using diffusion models for text generation. They propose a model that allows: "extreme flexibility", "exact sampling", and the possibility to cheaply evaluate the model using log-likelihood, using a Markov chain.

As stated before, they define a forward diffusion process, and then learn how to reverse this process (We called it backward in the Introcution, as used in [RDMS]).

$$q(x^{(0,...,T)})=q(x^{0})\Pi_{t=1}^{T}q(x^{(t)}|x^{(t-1)})$$

After having applied this noise on the data, we train a backward process that should  take $q(x^{(0,...,T)})$ as input data and retrieve $q(x^{0})$ through T steps too:

$$p(x^{(0,...,T)})=p(x^{T})\Pi_{t=1}^{T}p(x^{(t-1)}|x^{(t)})$$

We notice that this time we start at step T and go back through the steps ($x^{(t-1)}|x^{(t)}$) to retrieve original distribution.

The training is done by maximizing the log-likelihood of the model.
Amongst the serveral datasets used for evaluation, MNIST has been used. The log likelihood of the diffusion model reached 317 bits compared to 349 for the perfect model.

Following these promising results, the method has been readapted to several models. However, this model is still used on images rather than text.

# RDMs

RDM, standing for Reparameterized Discrete diffusion Models, presented in the paper [A Reparameterized Discrete Diffusion Model for Text Generation] by Zheng et al, argues that it exists and equivalent reparametrization of the sampling used previously in [Deep Unsupervised Learning using Nonequilibrium Thermodynamics] by Sohl-Dickstein et al.
This reparameterized sequence follows rules at each step (route-and-denoise process):
- has a probability of being denoised
- has a probability of being reseted to a noisy state following the routing mechanism

This process is used to noise directly the token embeddings of words, instead of adding it. It enables to stay in the token space to train the model. The route-denoise process allows to make the training process stable.

The forward process replaces tokens with other tokens from a noise distribution. A transition matrix stores the probability of each token to be corrupted into another token, and the denoising process rebuilds the orignal sentence.

The architecture used (cf annex F) predicts a target length. The input of the model is the timestep embedding (the sequence's current t variable, cf the froward/backward processes), the t is projected using sinusoidal embedinggs then passes through a two-layer MLP. We concatenate to this embedding the token embedding.

This input goes through a Transformer and then is projected to the vocabulary space to determine which word is predicted.

This work is important because it allows to use a continuous model on a discrete problem such as natural language processing. It allows to generate text too using a first sentence to which is appended noise.
